In [ ]:
# ============================================================
# CELL 1 — Mount Google Drive & Load Databases
# This connects Colab to your Google Drive so it can
# access sns_bert.db and sns_network.db
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import shutil
import os

# Copy databases from Drive to Colab's fast local storage
# Update this path if your folder name is different
DRIVE_PATH = '/content/drive/MyDrive/SNS_Capstone'

shutil.copy(f'{DRIVE_PATH}/sns_bert.db',    '/content/sns_bert.db')
shutil.copy(f'{DRIVE_PATH}/sns_network.db', '/content/sns_network.db')

# Confirm both files are there
for f in ['sns_bert.db', 'sns_network.db']:
    size = os.path.getsize(f'/content/{f}')
    print(f"✅ {f} loaded — {size:,} bytes")

In [ ]:
# ============================================================
# CELL 2 — Install Libraries
# These are the AI tools we need for BERT fine-tuning
# transformers = Hugging Face's BERT library
# torch = the deep learning engine underneath
# ============================================================

!pip install transformers torch scikit-learn pandas numpy -q

import torch
print(f"✅ PyTorch version  : {torch.__version__}")
print(f"✅ GPU available    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU name         : {torch.cuda.get_device_name(0)}")

In [ ]:
# ============================================================
# CELL 3 — Load & Prepare Data from sns_bert.db
# We pull the conversations and labels, then split into
# train / validation / test sets exactly as we defined
# when we built the database
# ============================================================

import sqlite3
import pandas as pd

conn = sqlite3.connect('/content/sns_bert.db')
df = pd.read_sql("""
    SELECT c.session_id,
           c.raw_text,
           ld.label,
           ld.pattern_type,
           ld.severity,
           ts.split
    FROM conversations c
    JOIN labeled_data ld ON c.conversation_id = ld.conversation_id
    JOIN training_splits ts ON c.conversation_id = ts.conversation_id
""", conn)
conn.close()

# Split into train / val / test
train_df = df[df['split'] == 'train'].reset_index(drop=True)
val_df   = df[df['split'] == 'val'].reset_index(drop=True)
test_df  = df[df['split'] == 'test'].reset_index(drop=True)

print("✅ Data loaded successfully!")
print(f"\n   Total conversations : {len(df)}")
print(f"   Train set           : {len(train_df)}")
print(f"   Validation set      : {len(val_df)}")
print(f"   Test set            : {len(test_df)}")
print(f"\n   Label distribution:")
print(df['label'].value_counts().to_string())
print(f"\n   Pattern breakdown:")
print(df['pattern_type'].value_counts().to_string())

In [ ]:
# ============================================================
# CELL 4 — Tokenize Data for BERT
# BERT can't read raw text — it needs numbers.
# The tokenizer converts each conversation into
# token IDs and attention masks that BERT understands.
# We cap at 512 tokens — BERT's maximum input length.
# ============================================================

from transformers import BertTokenizer
import torch
from torch.utils.data import Dataset, DataLoader

# Load the pre-trained BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
print("✅ BERT tokenizer loaded!")

# Custom Dataset class — wraps our conversations for PyTorch
class SNSDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=512):
        self.data      = dataframe
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text  = str(self.data['raw_text'].iloc[idx])
        label = int(self.data['label'].iloc[idx])

        encoding = self.tokenizer(
            text,
            max_length     = self.max_len,
            padding        = 'max_length',
            truncation     = True,
            return_tensors = 'pt'
        )
        return {
            'input_ids'      : encoding['input_ids'].squeeze(),
            'attention_mask' : encoding['attention_mask'].squeeze(),
            'label'          : torch.tensor(label, dtype=torch.long)
        }

# Create datasets
train_dataset = SNSDataset(train_df, tokenizer)
val_dataset   = SNSDataset(val_df,   tokenizer)
test_dataset  = SNSDataset(test_df,  tokenizer)

# Create DataLoaders — these feed data to BERT in batches
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=8)
test_loader  = DataLoader(test_dataset,  batch_size=8)

print(f"✅ Datasets tokenized!")
print(f"   Train batches      : {len(train_loader)}")
print(f"   Validation batches : {len(val_loader)}")
print(f"   Test batches       : {len(test_loader)}")

# Show what one tokenized sample looks like
sample = train_dataset[0]
print(f"\n   Sample input_ids shape      : {sample['input_ids'].shape}")
print(f"   Sample attention_mask shape : {sample['attention_mask'].shape}")
print(f"   Sample label                : {sample['label']}")

In [ ]:
# ============================================================
# CELL 5 — Load BERT Model & Configure Training
# BertForSequenceClassification is BERT with a
# classification head on top — perfect for our
# binary safe/unsafe detection task (num_labels=2)
# ============================================================

from transformers import BertForSequenceClassification
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

# Load pre-trained BERT with a classification head
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels = 2  # 0 = safe, 1 = unsafe
)

# Move model to GPU (Tesla T4)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f"✅ BERT model loaded and moved to: {device}")

# Optimizer — AdamW is the standard for BERT fine-tuning
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

# Training config
EPOCHS = 4  # 4 passes through the training data

# Learning rate scheduler — gradually reduces LR during training
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps   = total_steps // 10,
    num_training_steps = total_steps
)

print(f"✅ Optimizer configured!")
print(f"   Learning rate      : 2e-5")
print(f"   Epochs             : {EPOCHS}")
print(f"   Total train steps  : {total_steps}")
print(f"   Warmup steps       : {total_steps // 10}")
print(f"\n🚀 Ready to train!")

In [ ]:
# ============================================================
# CELL 6 — Train BERT
# This is the core training loop. For each epoch:
#   1. Feed conversations to BERT in batches
#   2. BERT makes predictions
#   3. We measure how wrong it was (loss)
#   4. AdamW nudges the weights to do better
#   5. Repeat until BERT learns the patterns
# ============================================================

import numpy as np
from sklearn.metrics import accuracy_score

def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss, all_preds, all_labels = 0, [], []

    for batch in loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids,
                       attention_mask=attention_mask,
                       labels=labels)

        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    accuracy = accuracy_score(all_labels, all_preds)
    return avg_loss, accuracy

def eval_epoch(model, loader, device):
    model.eval()
    total_loss, all_preds, all_labels = 0, [], []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)

            outputs = model(input_ids=input_ids,
                           attention_mask=attention_mask,
                           labels=labels)

            total_loss += outputs.loss.item()
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    accuracy = accuracy_score(all_labels, all_preds)
    return avg_loss, accuracy

# ── Run Training ──────────────────────────────────────────
print("=" * 55)
print("  SNS BERT Training — Starting")
print("=" * 55)

best_val_accuracy = 0
history = []

for epoch in range(1, EPOCHS + 1):
    print(f"\n  Epoch {epoch}/{EPOCHS}")
    print(f"  {'-'*40}")

    train_loss, train_acc = train_epoch(
        model, train_loader, optimizer, scheduler, device)
    val_loss, val_acc = eval_epoch(
        model, val_loader, device)

    history.append({
        'epoch': epoch,
        'train_loss': train_loss, 'train_acc': train_acc,
        'val_loss': val_loss,     'val_acc': val_acc
    })

    print(f"  Train  — Loss: {train_loss:.4f}  Acc: {train_acc:.4f}")
    print(f"  Val    — Loss: {val_loss:.4f}  Acc: {val_acc:.4f}")

    if val_acc > best_val_accuracy:
        best_val_accuracy = val_acc
        torch.save(model.state_dict(), '/content/sns_bert_best.pt')
        print(f"  💾 New best model saved! Val Acc: {val_acc:.4f}")

print(f"\n{'='*55}")
print(f"  Training Complete!")
print(f"  Best Validation Accuracy: {best_val_accuracy:.4f}")
print(f"{'='*55}")

In [ ]:
# ============================================================
# CELL 7 — Final Test Evaluation & Save Model to Drive
# We now test on completely unseen data and save
# the trained model back to Google Drive permanently
# ============================================================

from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Load the best saved model
model.load_state_dict(torch.load('/content/sns_bert_best.pt'))
print("✅ Best model loaded for final evaluation\n")

# Run on test set
test_loss, test_acc = eval_epoch(model, test_loader, device)

# Get detailed predictions for the report
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)
        outputs = model(input_ids=input_ids,
                       attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

print("=" * 55)
print("  SNS BERT — Final Test Results")
print("=" * 55)
print(f"\n  Test Accuracy : {test_acc:.4f}")
print(f"  Test Loss     : {test_loss:.4f}")
print(f"\n  Classification Report:")
print(classification_report(all_labels, all_preds,
      target_names=['Safe (0)', 'Unsafe (1)']))

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted Safe', 'Predicted Unsafe'],
            yticklabels=['Actually Safe', 'Actually Unsafe'])
plt.title('SNS BERT — Confusion Matrix')
plt.tight_layout()
plt.savefig('/content/sns_bert_confusion_matrix.png', dpi=150)
plt.show()
print("✅ Confusion matrix saved!")

# Save model to Google Drive permanently
import os
save_path = '/content/drive/MyDrive/SNS_Capstone/sns_bert_model'
os.makedirs(save_path, exist_ok=True)
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
shutil.copy('/content/sns_bert_confusion_matrix.png',
            '/content/drive/MyDrive/SNS_Capstone/')
print(f"✅ Model saved to Google Drive!")
print(f"   {save_path}")